In [23]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage, HumanMessage
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode

In [2]:
load_dotenv()

False

In [3]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage],add_messages]

In [4]:
@tool
def add(a: int,b: int):
    """ Addition function for two numbers """
    return a + b

In [5]:
@tool
def subtract(a: int,b: int):
    """ Subtraction function for two numbers """
    return a - b

In [6]:
@tool
def multiply(a: int, b: int):
    """ Multiplication function """
    return a * b

In [7]:
tools = [add,subtract,multiply]

In [9]:
model = ChatOllama(model='qwen.3:1.7b').bind_tools(tools)

In [27]:
def model_call(state: AgentState) -> AgentState:
    response = model.invoke(
        SystemMessage(content='You are my AI assistant, please answer my query to the best of your ability.'
                     )
        +
        state['messages']
    )
    return {"messages":[response]}

In [28]:
def should_continue(state: AgentState) -> str:
    messages = state['messages']
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

In [29]:
graph = StateGraph(AgentState)
graph.add_node("process",model_call)

In [30]:
tool_node = ToolNode(tools=tools)
graph.add_node("tools",tool_node)

In [31]:
graph.add_edge(START,"process")

In [32]:
graph.add_conditional_edges(
    "process",
    should_continue,
    {
        "end": END,
        "continue":"tools"
    }
)

In [33]:
graph.add_edge("tools","process")

In [34]:
app = graph.compile()

In [ ]:
ef print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

inputs = {"messages": [("user", "Add 40 + 12 and then multiply the result by 6. Also tell me a joke please.")]}
print_stream(app.stream(inputs, stream_mode="values"))